# Predictive Modeling of Quick-Commerce Delivery Delays Using Logistic Regression\n### Course: Fundamentals of Data Science (CSDS2002P) — UPES Dehradun\n**Author**: Anukalp Verma (SAP ID: 590028327, Batch B2 Data Science)\n\nThis notebook walks through an end-to-end binary classification project predicting whether an order breaches its 10-minute delivery SLA using **Logistic Regression**.

In [ ]:
import os\nimport numpy as np\nimport pandas as pd\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report\n\n# Load dataset\ndf = pd.read_csv('data/blinkit_orders_cleaned.csv')\nprint('Dataset Dimensions:', df.shape)\ndf.head()

## 1. Feature Engineering\nWe construct:\n- `order_hour`: Dispatch hour from order timestamp\n- `is_weekend`: Binary flag for weekend dispatch\n- `is_peak_hours`: Binary indicator for dinner peak rush (18:00 - 22:00)\n- `is_delayed`: Target label ($1 = \text{Delayed}, 0 = \text{On Time}$)

In [ ]:
df['order_datetime'] = pd.to_datetime(df['order_date'])\ndf['order_hour'] = df['order_datetime'].dt.hour\ndf['day_of_week'] = df['order_datetime'].dt.day_of_week\ndf['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)\ndf['is_peak_hours'] = df['order_hour'].isin([18, 19, 20, 21, 22]).astype(int)\ndf['is_delayed'] = (df['delivery_status'] != 'On Time').astype(int)\n\nprint('Class Distribution:')\nprint(df['is_delayed'].value_counts(normalize=True))\n\nfeature_cols = ['order_total', 'order_hour', 'is_weekend', 'is_peak_hours']\nX = pd.get_dummies(df[feature_cols + ['payment_method']], columns=['payment_method'], drop_first=True)\ny = df['is_delayed']\nX.head()

## 2. Train-Test Split and Feature Scaling\nLogistic Regression optimizes coefficients via gradient methods on log-loss. Standardization via `StandardScaler` ensures all features contribute fairly to the $L_2$ regularization penalty.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)\nscaler = StandardScaler()\nX_train_scaled = scaler.fit_transform(X_train)\nX_test_scaled = scaler.transform(X_test)\nprint('Train partition size:', X_train.shape[0])\nprint('Test partition size:', X_test.shape[0])

## 3. Training Logistic Regression with Balanced Class Weights\nBecause delays represent ~30% of orders, `class_weight='balanced'` assigns higher loss penalties to false negatives, ensuring the model identifies delays rather than defaulting to the majority class.

In [ ]:
clf = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)\nclf.fit(X_train_scaled, y_train)\n\ny_pred = clf.predict(X_test_scaled)\ny_prob = clf.predict_proba(X_test_scaled)[:, 1]\n\nprint('Model Evaluation Results:')\nprint(f'Accuracy:  {accuracy_score(y_test, y_pred)*100:.2f}%')\nprint(f'Precision: {precision_score(y_test, y_pred)*100:.2f}%')\nprint(f'Recall:    {recall_score(y_test, y_pred)*100:.2f}%')\nprint(f'F1-Score:  {f1_score(y_test, y_pred):.4f}')\nprint(f'ROC-AUC:   {roc_auc_score(y_test, y_prob):.4f}')\nprint('\nConfusion Matrix:\n', confusion_matrix(y_test, y_pred))\nprint('\nClassification Report:\n', classification_report(y_test, y_pred))

## 4. Odds Ratios (e^Beta)\nIn Logistic Regression, the exponentiated coefficient $\exp(\beta_j)$ gives the multiplicative change in the odds of delay for a 1-unit increase in the standardized feature.

In [ ]:
odds_df = pd.DataFrame({\n    'Feature': X.columns,\n    'Beta': clf.coef_[0].round(4),\n    'Odds_Ratio_exp_Beta': np.exp(clf.coef_[0]).round(4)\n}).sort_values('Odds_Ratio_exp_Beta', ascending=False).reset_index(drop=True)\nodds_df

## 5. Live Single-Order Delay Prediction\nWe test the model on a simulated high-value dinner-rush order versus an off-peak morning order.

In [ ]:
def predict_delay(cart_value, hour, weekend, cod_cash=1):\n    row = pd.DataFrame([{col: 0 for col in X.columns}])\n    row['order_total'] = cart_value\n    row['order_hour'] = hour\n    row['is_weekend'] = weekend\n    row['is_peak_hours'] = 1 if hour in [18, 19, 20, 21, 22] else 0\n    row['payment_method_Cash'] = cod_cash\n    prob = clf.predict_proba(scaler.transform(row))[0, 1]\n    return {'Probability': f'{prob*100:.2f}%', 'Predicted': 'Delayed' if prob >= 0.5 else 'On Time'}\n\nprint('Peak Dinner Rush Order (₹2,500 at 8 PM):', predict_delay(2500, 20, 0, 1))\nprint('Morning Off-Peak Order (₹450 at 10 AM):', predict_delay(450, 10, 1, 0))